# CSF matrix for the caterpillar basis

Functions related to the CSF matrix and the symmetric-tree subspace studied by González, Orellana, and Tomba.

## Setup

Load the core DNC routines and shared helper functions.

In [2]:
%run DNCfundamentals.ipynb
%run helperFunctions.ipynb

In [3]:
# Input: a valid leading partition (i.e., a non-hook partition)
# Output: the unique caterpillar with that leading partition that is part of the caterpillar-basis
#         introduced in the González-Orellana-Tomba project
def basis_caterpillar(leading):
    """Construct the caterpillar-basis tree with the requested leading partition."""
    G = Graph()
    internal_vertices = [0]
    
    if len(leading) == 1:
        G = G.disjoint_union(graphs.StarGraph(leading[0]-1), labels='integers')
        return G
    if leading[1] == 1:
        return None
    else:
        for i in range(1,len(leading)-1):
            internal_vertices.append(leading[i]+internal_vertices[i-1])

        internal_vertices.append(internal_vertices[-1]+leading[-1])

        for i in range(1,len(leading)):
            G = G.disjoint_union(graphs.StarGraph(leading[i]-1), labels='integers')
            
        G = G.disjoint_union(graphs.StarGraph(leading[0]-1), labels='integers')

        for i in range(0, len(internal_vertices)-1):
            G.add_edge([internal_vertices[i],internal_vertices[i+1]])
        # G.plot().show()
    return G

# Input: an integer n
# Output: a list containing all non-hook partitions of n
    
def leading_partition_list(n):
    """Return the non-hook partitions of ``n`` used to index the basis."""
    partitions_list = Partitions(n).list()
    leadings_list = []
    
    for p in partitions_list:
        if len(p) == 1:
            leadings_list.append(p)
        else:
            if p[1] != 1:
                leadings_list.append(p)
                
    return leadings_list

# Output: matrix in which each row contains the CSF vector of a caterpillar in the caterpillar basis
#        introduced by González-Orellana-Tomba
def basis_matrix(n):
    """Return the matrix whose rows are CSF vectors of the caterpillar basis."""
    tree_list = []
    partitions_list = Partitions(n).list()
    leadings_list = leading_partition_list(n)
    
    seen_list = {}
    
    L_P = len(partitions_list)
    M = np.zeros((0, L_P))
    
    for partition in leadings_list:
        T = basis_caterpillar(partition)
        tree_list.append(T)
        
    for i in range(0,len(tree_list)):
        tree_list[i] = tree_list[i].copy()
        # tree_list[i].plot().show()
        CSF_vector = CSF_helper(tree_list[i], len(tree_list[i].vertices()), seen_list)
        # print(CSF_vector)
        M = np.vstack((M, CSF_vector))
    return M

# Input: basismatrix is the CSF matrix where CSF vectors are written in rows
# Input: CSFvector is the CSF of the tree whose linear combination we are trying to find
# Output: vector of coefficients that determine the linear combination

def find_basis_coefficients(basismatrix, csfvector):
    """Express ``csfvector`` in the row basis of ``basismatrix``."""
    np.set_printoptions(suppress=True)
    # csfvector_transpose = csfvector.tranpose() # gives the csf vector as a column vector
    basismatrix_transpose = basismatrix.transpose() # this gives the CSF basis matrix with vectors in columns
    
    basis_mult = np.matmul(basismatrix, basismatrix_transpose)
    # print(basis_mult)
    
    rhs_mult = np.matmul(basismatrix, csfvector)
    # print(rhs_mult)
    
    basis_mult_inverse = np.linalg.inv(basis_mult)
    # print(basis_mult_inverse)
    
    coefficients = np.matmul(basis_mult_inverse, rhs_mult)
    
    return coefficients

## Exploratory computation

This cell enumerates trees and can become expensive as `n` grows. Start with a smaller value when testing the workflow.

In [18]:
n=11

tree_list = []
partitions_list = Partitions(n).list()
seen_list = {}
tree_iterator = graphs.trees(n)
    
for T in tree_iterator:
    tree_list.append(T)            
        
L_T = len(tree_list)
L_P = len(partitions_list)

for i in range(L_T):
    sumwithout1 = 0
    sumwith1 = 0
    tree_list[i] = tree_list[i].copy()
    CSF_vector = CSF_helper(tree_list[i], n, seen_list)
    l_p = get_leading_partition(CSF_vector, n)
    
    if count_ones(l_p)==1:
    
        for j in range(len(partitions_list)):
            if count_ones(partitions_list[j])==0 and len(partitions_list[j])==len(l_p)-1:
                sumwithout1 += CSF_vector[j]

            if count_ones(partitions_list[j])==1 and len(partitions_list[j])==len(l_p):
                sumwith1 += abs(CSF_vector[j])
        print(sumwithout1, '|', sumwith1, '| Internal edges:', len(l_p)-1)

2 | 1 | Internal edges: 5
2 | 1 | Internal edges: 5
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 3
2 | 1 | Internal edges: 3
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 3
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 3
3 | 2 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 3
2 | 1 | Internal edges: 3
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 3
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 3
2 | 1 | Internal edges: 5
2 | 1 | Internal edges: 5
3 | 2 | Internal edges: 5
3 | 2 | Internal edges: 4
3 | 2 | Internal edges: 4
3 | 2 | Internal edges: 5
3 | 2 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 4
2 | 1 | Internal edges: 3
3 | 2 | Internal edges: 4
3 | 2 | Inte